# Spotify Project Data Cleaning

**Student:** Min-Ju "Elena" Wei  
**Project:** Track Promotion Strategy  
**Branch:** cleaning-elenawei

## Project Vision

This project supports a music label marketing manager in comparing tracks within the same genre and release period. The goal is to identify musical characteristics associated with higher historical Spotify popularity and stream counts. The output supports promotional decisions but does not automatically predict whether a track will become a hit.

## Business Question

Within the same genre and release period, which musical characteristics are associated with higher Spotify track popularity and stream counts?

- Primary metric: `track_popularity`
- Secondary metric: `streams_total`

from pathlib import Path

import pandas as pd


DATA_PATH = Path("../data/raw/spotify_full.csv")

print("File exists:", DATA_PATH.exists())
print("File size (GB):", round(DATA_PATH.stat().st_size / 1_000_000_000, 2))
print("Pandas version:", pd.__version__)

In [1]:
from pathlib import Path

import pandas as pd


DATA_PATH = Path("../data/raw/spotify_full.csv")

print("File exists:", DATA_PATH.exists())
print("File size (GB):", round(DATA_PATH.stat().st_size / 1_000_000_000, 2))
print("Pandas version:", pd.__version__)

File exists: True
File size (GB): 0.72
Pandas version: 3.0.6


In [2]:
# Read only the first 5 rows to inspect the dataset safely
preview_df = pd.read_csv(
    DATA_PATH,
    nrows=5,
    encoding="cp1252"
)

print("Preview shape:", preview_df.shape)
display(preview_df)

print("\nColumn names:")
print(preview_df.columns.tolist())

Preview shape: (5, 88)


,track_id,spotify_id,title,title_normalized,track_artist,album_id,album_name,release_year,release_date,duration_ms,...,danceability_trend,decade_track_count,anomaly_flags,has_anomaly,release_year_precision_aug,Unnamed: 83,Unnamed: 84,Unnamed: 85,Unnamed: 86,Unnamed: 87
0,spotify:track:0yLdNVWF3Srea0uzk55zFn,0yLdNVWF3Srea0uzk55zFn,Flowers,flowers,Miley Cyrus,7I0tjwFtxUwBC1vgyeMAax,Flowers,2023,NaN,200455,...,0.549115,201603,NaN,False,False,NaN,NaN,NaN,NaN,NaN
1,spotify:track:0sf12qNH5qcw8qpgymFOqD,0sf12qNH5qcw8qpgymFOqD,Blinding Lights,blinding lights,The Weeknd,2ZfHkwHuoAZrlz7RMj0PDz,Blinding Lights,2019,11/29/2019,201573,...,0.542672,530477,NaN,False,False,NaN,NaN,NaN,NaN,NaN
2,spotify:track:21jGcNKet2qwijlDFuPiPb,21jGcNKet2qwijlDFuPiPb,Circles,circles,Post Malone,4g1ZRSobMefqF6nelkgibi,Hollywood's Bleeding,2019,9/6/2019,215280,...,0.542672,530477,NaN,False,False,NaN,NaN,NaN,NaN,NaN
3,spotify:track:2b8fOow8UzyDFAE27YhOZM,2b8fOow8UzyDFAE27YhOZM,Memories,memories,Maroon 5,3nR9B40hYLKLcR0Eph3Goc,Memories,2019,9/20/2019,189486,...,0.542672,530477,NaN,False,False,NaN,NaN,NaN,NaN,NaN
4,spotify:track:7k4t7uLgtOxPwTpFmtJNTY,7k4t7uLgtOxPwTpFmtJNTY,Tusa,tusa,KAROL G,7mKevNHhVnZER3BLgI8O4F,Tusa,2019,11/7/2019,200960,...,0.542672,530477,NaN,False,False,NaN,NaN,NaN,NaN,NaN



Column names:
['track_id', 'spotify_id', 'title', 'title_normalized', 'track_artist', 'album_id', 'album_name', 'release_year', 'release_date', 'duration_ms', 'explicit_flag', 'has_audio_features', 'source_dataset', 'energy', 'valence', 'danceability', 'acousticness', 'instrumentalness', 'speechiness', 'liveness', 'loudness', 'tempo', 'key', 'key_name', 'mode', 'mode_name', 'time_signature', 'genre_l1', 'genre_l2', 'genre_raw', 'track_popularity', 'streams_total', 'streams_source', 'streams_total_estimated', 'stream_density', 'chart_peak_position', 'weeks_on_chart', 'chart_date_first', 'popularity_imputed_aug', 'track_age_days', 'era', 'decade', 'energy_loudness_ratio', 'mood_index', 'acoustic_synthetic_ratio', 'emotional_arc', 'tempo_band', 'is_likely_instrumental', 'is_likely_live', 'is_likely_spoken_word', 'loudness_war_index', 'novelty_score', 'hit_probability', 'mood_cluster', 'taste_cluster', 'fp_energy', 'fp_valence', 'fp_danceability', 'fp_acousticness', 'fp_instrumentalness',

In [3]:
# Create a small raw sample for inspection
SAMPLE_FOLDER = Path("../data/sample")
SAMPLE_FOLDER.mkdir(parents=True, exist_ok=True)

# Keep all columns but read only the first 50 rows
raw_sample_df = pd.read_csv(
    DATA_PATH,
    nrows=50,
    encoding="cp1252"
)

RAW_SAMPLE_PATH = SAMPLE_FOLDER / "spotify_raw_sample_50.csv"

# Save with UTF-8 encoding for easier sharing
raw_sample_df.to_csv(
    RAW_SAMPLE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Sample created:", RAW_SAMPLE_PATH)
print("Sample shape:", raw_sample_df.shape)
print(
    "Sample file size (KB):",
    round(RAW_SAMPLE_PATH.stat().st_size / 1_000, 2)
)

Sample created: ..\data\sample\spotify_raw_sample_50.csv
Sample shape: (50, 88)
Sample file size (KB): 38.16


In [4]:
# Profile missing values across the full dataset in manageable chunks
chunk_size = 100_000
total_rows = 0
missing_counts = None

for chunk in pd.read_csv(
    DATA_PATH,
    encoding="latin-1",
    chunksize=chunk_size,
    low_memory=False
):
    total_rows += len(chunk)

    if missing_counts is None:
        missing_counts = chunk.isna().sum()
    else:
        missing_counts = missing_counts.add(
            chunk.isna().sum(),
            fill_value=0
        )

    print(f"Processed {total_rows:,} rows")

missing_summary = pd.DataFrame({
    "column": missing_counts.index,
    "missing_count": missing_counts.values
})

missing_summary["missing_percent"] = (
    missing_summary["missing_count"] / total_rows * 100
).round(2)

missing_summary = missing_summary.sort_values(
    "missing_percent",
    ascending=False
).reset_index(drop=True)

print(f"\nTotal rows: {total_rows:,}")
print(f"Total columns: {len(missing_counts)}")

display(missing_summary.head(25))

Processed 100,000 rows
Processed 200,000 rows
Processed 300,000 rows
Processed 400,000 rows
Processed 500,000 rows
Processed 600,000 rows
Processed 700,000 rows
Processed 800,000 rows
Processed 900,000 rows
Processed 1,000,000 rows
Processed 1,048,575 rows

Total rows: 1,048,575
Total columns: 88


,column,missing_count,missing_percent
0,Unnamed: 87,1048574,100.00
1,Unnamed: 86,1048571,100.00
2,Unnamed: 85,1048565,100.00
3,Unnamed: 84,1048543,100.00
4,Unnamed: 83,1048454,99.99
5,taste_cluster,1036733,98.87
6,chart_peak_position,1034045,98.61
7,weeks_on_chart,1033560,98.57
8,streams_source,1029446,98.18
9,album_id,1025167,97.77


In [5]:
# Inspect rows containing values in the unexpected Unnamed columns
unnamed_columns = [
    "Unnamed: 83",
    "Unnamed: 84",
    "Unnamed: 85",
    "Unnamed: 86",
    "Unnamed: 87"
]

columns_to_check = [
    "track_id",
    "spotify_id",
    "title",
    "track_artist"
] + unnamed_columns

unexpected_rows = []

for chunk in pd.read_csv(
    DATA_PATH,
    encoding="latin-1",
    chunksize=100_000,
    low_memory=False,
    usecols=columns_to_check
):
    has_unexpected_value = chunk[unnamed_columns].notna().any(axis=1)

    if has_unexpected_value.any():
        selected_rows = chunk.loc[
            has_unexpected_value,
            columns_to_check
        ].copy()

        selected_rows.insert(
            0,
            "csv_row_number",
            selected_rows.index + 2
        )

        unexpected_rows.append(selected_rows)

unexpected_rows_df = pd.concat(
    unexpected_rows,
    ignore_index=True
)

print(
    "Rows with values in Unnamed columns:",
    len(unexpected_rows_df)
)

display(unexpected_rows_df.head(20))

Rows with values in Unnamed columns: 126


,csv_row_number,track_id,spotify_id,title,track_artist,Unnamed: 83,Unnamed: 84,Unnamed: 85,Unnamed: 86,Unnamed: 87
0,4260,spotify:track:4wopYJ5wpYOM6ogm2Jugnw,4wopYJ5wpYOM6ogm2Jugnw,"fuck, i'm lonely (with Anne-Marie) - from ??3 ...",december 1963 oh what a night,False,NaN,NaN,NaN,NaN
1,7038,spotify:track:1gACe11pZiy8Xv3SY0ocyz,1gACe11pZiy8Xv3SY0ocyz,"Heartbeat (From the ?Lyle, Lyle, Crocodile??O...","Trinidad James & ScHoolboy Q)""",False,False,NaN,NaN,NaN
2,86716,spotify:track:1YfDfk1s3U1iexGila4UC2,1YfDfk1s3U1iexGila4UC2,"èªªæ£å°±æ£ - ""?ä»»3: ?è??ä»»"" ?»å½±ä¸»é??...",super rare feat wonstein ph1,False,NaN,NaN,NaN,NaN
3,90501,spotify:track:3pzrjz10VPZWaZJeMMaMIS,3pzrjz10VPZWaZJeMMaMIS,"?ç??¢é?ä¹æ¯??- ?é? ""?è­¦"" ?å°¾??,,Alfred...","No.4: I. Adagio""",False,False,NaN,NaN,NaN
4,102841,spotify:track:2iEtlaAIaHUDnsCnUHiVkV,2iEtlaAIaHUDnsCnUHiVkV,"Long Hairdryer, FÃ¶n, Baby Sleeping Sound, Haa...",1000 years,False,NaN,NaN,NaN,NaN
5,110635,spotify:track:3WUgND9BFNDFEbkCB9AzHg,3WUgND9BFNDFEbkCB9AzHg,"Schumann: Kinderszenen, Op. 15 - 7. æ¢¦å¹»??,s...",nocturnes op 27 no 2 in d flat major lento sos...,False,NaN,NaN,NaN,NaN
6,123631,spotify:track:6JSJfhvU7uI2kO8JLrxyP6,6JSJfhvU7uI2kO8JLrxyP6,"Mystery (Rosary) Sonata No. 2, ""The Visitation...","na fazenda (Casinha de sapÃª) - Ao vivo""",False,False,NaN,NaN,NaN
7,124257,spotify:track:5kcvIemT29FwXaHvp6W5iA,5kcvIemT29FwXaHvp6W5iA,"?ªä?åº¦é?æ¼«é·æ­²æ? - ""?ªå??±å°¼åº¦é?æ¼«é·æ­...",Ithaca,NaN,False,False,NaN,NaN
8,128609,spotify:track:0BQ3kSfeLFq3CMtuHbYUHD,0BQ3kSfeLFq3CMtuHbYUHD,"ë´ë?, ë²ê? ê·¸ë¦¬ê³???,,Epitone Project,,,20...","Act I: Dance of the Blessed Spirits""",False,False,NaN,NaN,NaN
9,130516,spotify:track:4kWOErfRKsu6Kzs96wv1fI,4kWOErfRKsu6Kzs96wv1fI,"æ¬²è??æ­¢ - ?é? ""æºå?é¢¨æ´3"" ?å°¾??,3,HAN...",on noublie jamais rien on vit avec,False,NaN,NaN,NaN,NaN


In [6]:
# Audit key fields across the full dataset
audit_columns = [
    "spotify_id",
    "title",
    "track_artist",
    "release_year",
    "duration_ms",
    "energy",
    "valence",
    "danceability",
    "acousticness",
    "instrumentalness",
    "speechiness",
    "liveness",
    "loudness",
    "tempo",
    "track_popularity",
    "streams_total"
]

bad_data_indices = set(
    unexpected_rows_df["csv_row_number"] - 2
)

missing_key_counts = pd.Series(0, index=audit_columns, dtype="int64")
seen_spotify_ids = set()
duplicate_spotify_ids = 0

invalid_counts = {
    "release_year_outside_1900_2026": 0,
    "duration_ms_not_positive": 0,
    "energy_outside_0_1": 0,
    "valence_outside_0_1": 0,
    "danceability_outside_0_1": 0,
    "acousticness_outside_0_1": 0,
    "instrumentalness_outside_0_1": 0,
    "speechiness_outside_0_1": 0,
    "liveness_outside_0_1": 0,
    "tempo_not_positive": 0,
    "track_popularity_outside_0_100": 0,
    "streams_total_negative": 0
}

valid_rows_checked = 0

for chunk in pd.read_csv(
    DATA_PATH,
    encoding="latin-1",
    chunksize=100_000,
    low_memory=False,
    usecols=audit_columns
):
    # Exclude the 126 malformed rows identified earlier
    chunk = chunk.loc[~chunk.index.isin(bad_data_indices)]
    valid_rows_checked += len(chunk)

    # Convert expected numeric columns to numeric values
    numeric_columns = [
        "release_year",
        "duration_ms",
        "energy",
        "valence",
        "danceability",
        "acousticness",
        "instrumentalness",
        "speechiness",
        "liveness",
        "loudness",
        "tempo",
        "track_popularity",
        "streams_total"
    ]

    chunk[numeric_columns] = chunk[numeric_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    missing_key_counts = missing_key_counts.add(
        chunk.isna().sum(),
        fill_value=0
    )

    for spotify_id in chunk["spotify_id"].dropna():
        if spotify_id in seen_spotify_ids:
            duplicate_spotify_ids += 1
        else:
            seen_spotify_ids.add(spotify_id)

    invalid_counts["release_year_outside_1900_2026"] += (
        chunk["release_year"].notna()
        & ~chunk["release_year"].between(1900, 2026)
    ).sum()

    invalid_counts["duration_ms_not_positive"] += (
        chunk["duration_ms"].notna()
        & (chunk["duration_ms"] <= 0)
    ).sum()

    for column in [
        "energy",
        "valence",
        "danceability",
        "acousticness",
        "instrumentalness",
        "speechiness",
        "liveness"
    ]:
        invalid_counts[f"{column}_outside_0_1"] += (
            chunk[column].notna()
            & ~chunk[column].between(0, 1)
        ).sum()

    invalid_counts["tempo_not_positive"] += (
        chunk["tempo"].notna()
        & (chunk["tempo"] <= 0)
    ).sum()

    invalid_counts["track_popularity_outside_0_100"] += (
        chunk["track_popularity"].notna()
        & ~chunk["track_popularity"].between(0, 100)
    ).sum()

    invalid_counts["streams_total_negative"] += (
        chunk["streams_total"].notna()
        & (chunk["streams_total"] < 0)
    ).sum()

print(f"Valid rows checked: {valid_rows_checked:,}")
print(f"Duplicate spotify_id values: {duplicate_spotify_ids:,}")

print("\nMissing values in key columns:")
display(
    missing_key_counts
    .sort_values(ascending=False)
    .rename("missing_count")
    .to_frame()
)

print("\nInvalid value counts:")
display(
    pd.Series(invalid_counts)
    .rename("invalid_count")
    .to_frame()
)

Valid rows checked: 1,048,449
Duplicate spotify_id values: 1

Missing values in key columns:


,missing_count
streams_total,9616
duration_ms,9612
release_year,9037
track_artist,8499
track_popularity,617
tempo,611
danceability,40
valence,40
speechiness,40
liveness,40



Invalid value counts:


,invalid_count
release_year_outside_1900_2026,588
duration_ms_not_positive,0
energy_outside_0_1,5
valence_outside_0_1,6
danceability_outside_0_1,6
acousticness_outside_0_1,5
instrumentalness_outside_0_1,6
speechiness_outside_0_1,581
liveness_outside_0_1,9570
tempo_not_positive,1066


In [7]:
# Investigate patterns behind invalid numeric values
investigation_columns = [
    "spotify_id",
    "title",
    "track_artist",
    "source_dataset",
    "release_year",
    "energy",
    "valence",
    "danceability",
    "acousticness",
    "instrumentalness",
    "speechiness",
    "liveness",
    "tempo",
    "track_popularity",
    "streams_total"
]

invalid_examples = []
source_summaries = []

for chunk in pd.read_csv(
    DATA_PATH,
    encoding="latin-1",
    chunksize=100_000,
    low_memory=False,
    usecols=investigation_columns
):
    chunk = chunk.loc[~chunk.index.isin(bad_data_indices)].copy()

    numeric_columns = [
        "release_year",
        "energy",
        "valence",
        "danceability",
        "acousticness",
        "instrumentalness",
        "speechiness",
        "liveness",
        "tempo",
        "track_popularity",
        "streams_total"
    ]

    chunk[numeric_columns] = chunk[numeric_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    audio_columns = [
        "energy",
        "valence",
        "danceability",
        "acousticness",
        "instrumentalness",
        "speechiness",
        "liveness"
    ]

    chunk["invalid_release_year"] = (
        chunk["release_year"].notna()
        & ~chunk["release_year"].between(1900, 2026)
    )

    chunk["invalid_audio_feature"] = False

    for column in audio_columns:
        chunk["invalid_audio_feature"] |= (
            chunk[column].notna()
            & ~chunk[column].between(0, 1)
        )

    chunk["invalid_tempo"] = (
        chunk["tempo"].notna()
        & (chunk["tempo"] <= 0)
    )

    chunk["invalid_popularity"] = (
        chunk["track_popularity"].notna()
        & ~chunk["track_popularity"].between(0, 100)
    )

    chunk["invalid_streams"] = (
        chunk["streams_total"].notna()
        & (chunk["streams_total"] < 0)
    )

    flag_columns = [
        "invalid_release_year",
        "invalid_audio_feature",
        "invalid_tempo",
        "invalid_popularity",
        "invalid_streams"
    ]

    invalid_mask = chunk[flag_columns].any(axis=1)

    source_summary = (
        chunk.groupby("source_dataset", dropna=False)[flag_columns]
        .sum()
    )
    source_summaries.append(source_summary)

    if invalid_mask.any():
        example_columns = [
            "spotify_id",
            "title",
            "source_dataset",
            "release_year",
            "liveness",
            "tempo",
            "track_popularity",
            "streams_total"
        ] + flag_columns

        invalid_examples.append(
            chunk.loc[invalid_mask, example_columns].head(10)
        )

invalid_by_source = (
    pd.concat(source_summaries)
    .groupby(level=0)
    .sum()
)

print("Invalid values by source dataset:")
display(invalid_by_source)

print("\nExample invalid records:")
display(
    pd.concat(invalid_examples, ignore_index=True).head(10)
)

Invalid values by source dataset:


,invalid_release_year,invalid_audio_feature,invalid_tempo,invalid_popularity,invalid_streams
source_dataset,,,,,
-3.676,1,1,0,0,0
0,1,1,0,0,0
0.000268,0,1,0,1,0
0.00107,0,1,0,1,0
0.00536,0,1,0,1,0
...,...,...,...,...,...
5.448944,1,1,0,0,0
7,0,1,0,0,0
FALSE,1,0,0,0,0



Example invalid records:


,spotify_id,title,source_dataset,release_year,liveness,tempo,track_popularity,streams_total,invalid_release_year,invalid_audio_feature,invalid_tempo,invalid_popularity,invalid_streams
0,322tcqPhma7F6hfCeLisx0,"fuck, i'm lonely (with Anne-Marie) - from ??3 ...",0.806,NaN,9.000,1.0,NaN,20682.316,False,True,False,False,False
1,10zz9RZt9DnqcxNWksRNrx,?ªç£?®è±??,0.939,NaN,-3.498,2.0,9.355561e+06,NaN,False,True,False,True,False
2,5LisHkYlpdkqNYl1HZ5cKd,Good Girl Gone Missin??good girl gone missin,0.492,NaN,-7.604,7.0,3.801113e+07,NaN,False,True,False,True,False
3,4NaaF28BeO9WzjDrSS71Nz,??³ã¹?ã¼??,0.695,NaN,-3.870,7.0,3.903627e+06,NaN,False,True,False,True,False
4,3dPtXHP0oXQ4HCWHsOA9js,å¤ã«é§ã???,0.874,NaN,-5.221,8.0,1.435738e+08,NaN,False,True,False,True,False
5,6sWd8ZAH5bWWyjwb7ZV2Zb,LALALAY ? ë¼ë¦?lalalay,0.616,174880.0,110.016,NaN,NaN,NaN,True,True,False,False,False
6,6YCwX5jlAEFL775YJqgLqN,?¾ã¶??,0.725,NaN,-5.247,7.0,3.261032e+06,NaN,False,True,False,True,False
7,1di1C0QI6Y92yZPYn6XYAZ,?·ã«?¨ã???,0.934,NaN,-2.685,2.0,2.361823e+06,NaN,False,True,False,True,False
8,1dBroK67DSaaEE5kvkVjzG,ä½å??èªªè©?,0.389,NaN,-13.106,5.0,2.391768e+06,NaN,False,True,False,True,False
9,2PlbwoIVr25oita35jg9ht,?±å???,0.789,NaN,-7.634,7.0,3.285185e+06,NaN,False,True,False,True,False


In [8]:
# Count valid, missing, and unexpected source_dataset values
valid_sources = {
    "arc8_spotify_1m",
    "tidytuesday_spotify"
}

valid_source_rows = 0
missing_source_rows = 0
unexpected_source_rows = 0

for chunk in pd.read_csv(
    DATA_PATH,
    encoding="latin-1",
    chunksize=100_000,
    low_memory=False,
    usecols=["source_dataset"]
):
    valid_source_rows += chunk["source_dataset"].isin(
        valid_sources
    ).sum()

    missing_source_rows += chunk["source_dataset"].isna().sum()

    unexpected_source_rows += (
        chunk["source_dataset"].notna()
        & ~chunk["source_dataset"].isin(valid_sources)
    ).sum()

print(f"Valid source rows: {valid_source_rows:,}")
print(f"Missing source rows: {missing_source_rows:,}")
print(f"Unexpected source rows: {unexpected_source_rows:,}")
print(
    "Total checked:",
    f"{valid_source_rows + missing_source_rows + unexpected_source_rows:,}"
)

Valid source rows: 1,038,824
Missing source rows: 37
Unexpected source rows: 9,714
Total checked: 1,048,575


In [9]:
# Identify and inspect duplicate spotify_id records
seen_ids = set()
duplicate_ids = set()

for chunk in pd.read_csv(
    DATA_PATH,
    encoding="latin-1",
    chunksize=100_000,
    low_memory=False,
    usecols=["spotify_id", "source_dataset"]
):
    valid_chunk = chunk[
        chunk["source_dataset"].isin(valid_sources)
    ]

    for spotify_id in valid_chunk["spotify_id"].dropna():
        if spotify_id in seen_ids:
            duplicate_ids.add(spotify_id)
        else:
            seen_ids.add(spotify_id)

print("Number of duplicate Spotify IDs:", len(duplicate_ids))

columns_to_compare = [
    "track_id",
    "spotify_id",
    "title",
    "track_artist",
    "release_year",
    "duration_ms",
    "track_popularity",
    "streams_total",
    "source_dataset"
]

duplicate_records = []

if duplicate_ids:
    for chunk in pd.read_csv(
        DATA_PATH,
        encoding="latin-1",
        chunksize=100_000,
        low_memory=False,
        usecols=columns_to_compare
    ):
        matches = chunk[
            chunk["spotify_id"].isin(duplicate_ids)
            & chunk["source_dataset"].isin(valid_sources)
        ]

        if not matches.empty:
            duplicate_records.append(matches)

    duplicate_records_df = pd.concat(
        duplicate_records,
        ignore_index=True
    )

    display(duplicate_records_df)

else:
    duplicate_records_df = pd.DataFrame(
        columns=columns_to_compare
    )
    print(
        "No duplicate Spotify IDs remain "
        "after valid-source filtering."
    )

Number of duplicate Spotify IDs: 0
No duplicate Spotify IDs remain after valid-source filtering.


In [10]:
# Create the cleaned full dataset and a 50-row cleaned sample
PROCESSED_FOLDER = Path("../data/processed")
PROCESSED_FOLDER.mkdir(parents=True, exist_ok=True)

CLEANED_DATA_PATH = PROCESSED_FOLDER / "spotify_cleaned.csv"
CLEANED_SAMPLE_PATH = SAMPLE_FOLDER / "spotify_cleaned_sample_50.csv"

valid_sources = {
    "arc8_spotify_1m",
    "tidytuesday_spotify"
}

numeric_columns = [
    "release_year",
    "duration_ms",
    "energy",
    "valence",
    "danceability",
    "acousticness",
    "instrumentalness",
    "speechiness",
    "liveness",
    "loudness",
    "tempo",
    "track_popularity",
    "streams_total"
]

input_rows = 0
removed_invalid_source = 0
removed_missing_identifier = 0
output_rows = 0
first_chunk = True
sample_pieces = []

for chunk_number, chunk in enumerate(
    pd.read_csv(
        DATA_PATH,
        encoding="utf-8",
        encoding_errors="replace",
        chunksize=100_000,
        low_memory=False
    )
):
    input_rows += len(chunk)

    # Remove rows with shifted or corrupted fields
    valid_source_mask = chunk["source_dataset"].isin(valid_sources)
    removed_invalid_source += (~valid_source_mask).sum()
    clean_chunk = chunk.loc[valid_source_mask].copy()

    # Remove unexpected trailing columns
    unnamed_columns = [
        column
        for column in clean_chunk.columns
        if column.startswith("Unnamed:")
    ]
    clean_chunk = clean_chunk.drop(columns=unnamed_columns)

    # Remove extra spaces from text values
    text_columns = clean_chunk.select_dtypes(
        include=["object", "string"]
    ).columns

    for column in text_columns:
        clean_chunk[column] = clean_chunk[column].map(
            lambda value: (
                value.strip()
                if isinstance(value, str)
                else value
            )
        )

    # Convert empty strings to missing values
    clean_chunk[text_columns] = clean_chunk[text_columns].replace(
        "",
        pd.NA
    )

    # Ensure important numeric columns are numeric
    clean_chunk[numeric_columns] = clean_chunk[numeric_columns].apply(
        pd.to_numeric,
        errors="coerce"
    )

    # spotify_id and title are required identifiers
    missing_identifier_mask = (
        clean_chunk["spotify_id"].isna()
        | clean_chunk["title"].isna()
    )

    removed_missing_identifier += missing_identifier_mask.sum()
    clean_chunk = clean_chunk.loc[~missing_identifier_mask].copy()

    output_rows += len(clean_chunk)

    # Save the cleaned full dataset chunk by chunk
    clean_chunk.to_csv(
        CLEANED_DATA_PATH,
        mode="w" if first_chunk else "a",
        header=first_chunk,
        index=False,
        encoding="utf-8"
    )

    first_chunk = False

    # Collect a reproducible sample from each chunk
    sample_size = min(5, len(clean_chunk))

    sample_pieces.append(
        clean_chunk.sample(
            n=sample_size,
            random_state=42 + chunk_number
        )
    )

    print(
        f"Processed {input_rows:,} input rows; "
        f"kept {output_rows:,} cleaned rows"
    )

cleaned_sample_df = (
    pd.concat(sample_pieces, ignore_index=True)
    .sample(n=50, random_state=42)
    .reset_index(drop=True)
)

cleaned_sample_df.to_csv(
    CLEANED_SAMPLE_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nCleaning completed")
print(f"Input rows: {input_rows:,}")
print(f"Removed invalid-source rows: {removed_invalid_source:,}")
print(
    "Removed rows missing spotify_id or title:",
    f"{removed_missing_identifier:,}"
)
print(f"Cleaned rows: {output_rows:,}")
print(f"Cleaned columns: {cleaned_sample_df.shape[1]}")
print(f"Cleaned sample shape: {cleaned_sample_df.shape}")
print("Cleaned full dataset:", CLEANED_DATA_PATH)
print("Cleaned sample:", CLEANED_SAMPLE_PATH)

Processed 100,000 input rows; kept 99,435 cleaned rows
Processed 200,000 input rows; kept 198,693 cleaned rows
Processed 300,000 input rows; kept 297,960 cleaned rows
Processed 400,000 input rows; kept 397,155 cleaned rows
Processed 500,000 input rows; kept 496,316 cleaned rows
Processed 600,000 input rows; kept 595,367 cleaned rows
Processed 700,000 input rows; kept 694,288 cleaned rows
Processed 800,000 input rows; kept 792,998 cleaned rows
Processed 900,000 input rows; kept 891,677 cleaned rows
Processed 1,000,000 input rows; kept 990,537 cleaned rows
Processed 1,048,575 input rows; kept 1,038,823 cleaned rows

Cleaning completed
Input rows: 1,048,575
Removed invalid-source rows: 9,751
Removed rows missing spotify_id or title: 1
Cleaned rows: 1,038,823
Cleaned columns: 83
Cleaned sample shape: (50, 83)
Cleaned full dataset: ..\data\processed\spotify_cleaned.csv
Cleaned sample: ..\data\sample\spotify_cleaned_sample_50.csv


In [11]:
# Validate the cleaned full dataset
validation_rows = 0
validation_columns = None
unexpected_columns_count = 0
invalid_source_count = 0
missing_identifier_count = 0
duplicate_id_count = 0
invalid_numeric_count = 0
seen_clean_ids = set()

audio_columns = [
    "energy",
    "valence",
    "danceability",
    "acousticness",
    "instrumentalness",
    "speechiness",
    "liveness"
]

for chunk in pd.read_csv(
    CLEANED_DATA_PATH,
    encoding="utf-8",
    chunksize=100_000,
    low_memory=False
):
    validation_rows += len(chunk)

    if validation_columns is None:
        validation_columns = chunk.columns.tolist()

    unexpected_columns_count += sum(
        column.startswith("Unnamed:")
        for column in chunk.columns
    )

    invalid_source_count += (
        ~chunk["source_dataset"].isin(valid_sources)
    ).sum()

    missing_identifier_count += (
        chunk["spotify_id"].isna()
        | chunk["title"].isna()
    ).sum()

    for spotify_id in chunk["spotify_id"]:
        if spotify_id in seen_clean_ids:
            duplicate_id_count += 1
        else:
            seen_clean_ids.add(spotify_id)

    for column in audio_columns:
        invalid_numeric_count += (
            chunk[column].notna()
            & ~chunk[column].between(0, 1)
        ).sum()

    invalid_numeric_count += (
        chunk["track_popularity"].notna()
        & ~chunk["track_popularity"].between(0, 100)
    ).sum()

    invalid_numeric_count += (
        chunk["duration_ms"].notna()
        & (chunk["duration_ms"] <= 0)
    ).sum()

    invalid_numeric_count += (
        chunk["tempo"].notna()
        & (chunk["tempo"] <= 0)
    ).sum()

    invalid_numeric_count += (
        chunk["streams_total"].notna()
        & (chunk["streams_total"] < 0)
    ).sum()

validation_results = pd.DataFrame({
    "check": [
        "Row count matches cleaning output",
        "Column count equals 83",
        "No Unnamed columns",
        "Only valid source datasets",
        "No missing spotify_id or title",
        "No duplicate spotify_id",
        "No invalid core numeric values"
    ],
    "actual": [
        validation_rows,
        len(validation_columns),
        unexpected_columns_count,
        invalid_source_count,
        missing_identifier_count,
        duplicate_id_count,
        invalid_numeric_count
    ],
    "expected": [
        output_rows,
        83,
        0,
        0,
        0,
        0,
        0
    ]
})

validation_results["passed"] = (
    validation_results["actual"]
    == validation_results["expected"]
)

display(validation_results)

assert validation_results["passed"].all()

print("All validation checks passed.")

,check,actual,expected,passed
0,Row count matches cleaning output,1038823,1038823,True
1,Column count equals 83,83,83,True
2,No Unnamed columns,0,0,True
3,Only valid source datasets,0,0,True
4,No missing spotify_id or title,0,0,True
5,No duplicate spotify_id,0,0,True
6,No invalid core numeric values,0,0,True


All validation checks passed.


In [12]:
# Five record-level checks required for Homework 6
from pathlib import Path
import pandas as pd

current_folder = Path.cwd()
project_root = (
    current_folder.parent
    if current_folder.name == "notebooks"
    else current_folder
)

raw_path = project_root / "data" / "raw" / "spotify_full.csv"
cleaned_path = project_root / "data" / "processed" / "spotify_cleaned.csv"

record_checks = {
    "0yLdNVWF3Srea0uzk55zFn": "Preserved",
    "4wopYJ5wpYOM6ogm2Jugnw": "Removed",
    "1gACe11pZiy8Xv3SY0ocyz": "Removed",
    "1YfDfk1s3U1iexGila4UC2": "Removed",
    "3pzrjz10VPZWaZJeMMaMIS": "Removed",
}

target_ids = set(record_checks)
original_records = []

for chunk in pd.read_csv(
    raw_path,
    encoding="utf-8",
    encoding_errors="replace",
    chunksize=100_000,
    low_memory=False,
    usecols=["spotify_id", "title", "source_dataset"]
):
    matches = chunk[chunk["spotify_id"].isin(target_ids)]

    if not matches.empty:
        original_records.append(matches)

original_df = pd.concat(
    original_records,
    ignore_index=True
)

cleaned_ids = set()

for chunk in pd.read_csv(
    cleaned_path,
    chunksize=100_000,
    low_memory=False,
    usecols=["spotify_id"]
):
    cleaned_ids.update(
        chunk.loc[
            chunk["spotify_id"].isin(target_ids),
            "spotify_id"
        ].dropna()
    )

check_rows = []

for spotify_id, expected_action in record_checks.items():
    original_match = original_df[
        original_df["spotify_id"] == spotify_id
    ]

    if original_match.empty:
        original_value = "Record not found in raw data"
    else:
        row = original_match.iloc[0]
        original_value = (
            f"title={row['title']}; "
            f"source_dataset={row['source_dataset']}"
        )

    actual_action = (
        "Preserved"
        if spotify_id in cleaned_ids
        else "Removed"
    )

    check_rows.append({
        "spotify_id": spotify_id,
        "original": original_value,
        "expected": expected_action,
        "actual": actual_action,
        "passed": expected_action == actual_action
    })

record_check_df = pd.DataFrame(check_rows)

display(record_check_df)
print(
    "Record checks passed:",
    record_check_df["passed"].sum(),
    "of",
    len(record_check_df)
)

,spotify_id,original,expected,actual,passed
0,0yLdNVWF3Srea0uzk55zFn,title=Flowers; source_dataset=arc8_spotify_1m,Preserved,Preserved,True
1,4wopYJ5wpYOM6ogm2Jugnw,"title=fuck, i'm lonely (with Anne-Marie) - fro...",Removed,Removed,True
2,1gACe11pZiy8Xv3SY0ocyz,"title=Heartbeat (From the ?�Lyle, Lyle, Crocod...",Removed,Removed,True
3,1YfDfk1s3U1iexGila4UC2,"title=說散就散 - ""?�任3: ?��??�任"" ?�影主�???,3,TIA RA...",Removed,Removed,True
4,3pzrjz10VPZWaZJeMMaMIS,"title=?��??��?也是??- ?��? ""?�警"" ?�尾??,,Alfred H...",Removed,Removed,True


Record checks passed: 5 of 5
